# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asraserver06/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Classification**

I'm predicting whether a content page is "declining" or not — a binary (yes/no) label. This is classification because the output is one of two fixed categories, not a score or ranking, and not unlabeled groups (which would be clustering).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**
This label comes from an OBSERVED outcome, not a defined rule — it's taken directly from what actually happened in `trend_direction`: `is_declining_label = (trend_direction == "down")`. It's not a proxy, it's a real observed value.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision/Recall (or ROC-AUC), reported against the base rate**

The actual class split is close to balanced (~54% declining vs ~46% not declining), so accuracy alone wouldn't be as misleading here as I first assumed — but I'm still reporting precision/recall or ROC-AUC because the real-world cost of a false negative (missing a page that's actually declining) is different from a false positive (flagging a stable page unnecessarily), and a single accuracy number hides that distinction.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [21]:
!git clone https://github.com/asraserver06/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter
!ls

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 121 (delta 37), reused 104 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 1.82 MiB | 7.02 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


**Unit of analysis:** One row = one content page (a specific webpage belonging to one of FlyRank's clients). The dataset has 30,000 rows, each representing a distinct page. The target (`is_declining_label`) splits roughly evenly: ~54% declining, ~46% not declining.

In [22]:
!pwd
!ls

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [23]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df[['trend_direction', 'is_declining_label']].head(10)
df['is_declining_label'].value_counts(normalize=True)

,proportion
is_declining_label,
1,0.542067
0,0.457933


In [24]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Why ML beats a fixed rule here**

A simple if-statement rule (e.g. "if avg_position dropped by X, flag as declining") would only look at one signal at a time. But this dataset has 44 columns — search_volume, competition, CTR, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, word_count, content_type, and more — and a page's decline is likely driven by some *combination* of these interacting together, not any single threshold. For example, a drop in avg_position might matter a lot for a high-competition page but barely matter for a low-competition one. A fixed rule can't capture that kind of conditional, tangled interaction across many signals — that's exactly the case where ML earns its place over a hand-written rule.

In [26]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.